In [1]:
import os
import copy

from itertools import chain, combinations

from sklearn import linear_model
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score

import pandas as pd
import numpy as np

import torch
import tensorkrowch as tk
from tensorkrowch.decompositions import tt_rss
import matplotlib.pyplot as plt

from interpretability import *

## Train LR model

In [2]:
cwd = os.path.join(os.getcwd(), '..')

model_name = 'llr6'
n_classes = 2

In [3]:
# Load data
# ---------
featuresNA = ['TMB', 'Albumin', 'NLR', 'Age', 'Systemic_therapy_history',
                'CancerType1', 'CancerType2', 'CancerType3', 'CancerType4',
                'CancerType5', 'CancerType6', 'CancerType7', 'CancerType8',
                'CancerType9', 'CancerType10', 'CancerType11', 'CancerType12',
                'CancerType13', 'CancerType14', 'CancerType15', 'CancerType16']
phenoNA = 'Response'

numeric_featuresNA = ['TMB', 'Albumin', 'NLR', 'Age']

datasets = ['Chowell_train', 'Chowell_test', 'MSK1', 'MSK2', 'Shim_NSCLC',
            'Kato_panCancer', 'Vanguri_NSCLC', 'Ravi_NSCLC', 'Pradat_panCancer']

# scaler_type = 'MinMax'
scaler_type = 'Standard'

dataset = 'Chowell_train'
data_train, scalers_train = load_data(cwd,
                                      featuresNA,
                                      phenoNA,
                                      dataset,
                                      scaler_type)

dataset = 'Chowell_test'
data_test, scalers_test = load_data(cwd,
                                    featuresNA,
                                    phenoNA,
                                    dataset,
                                    scaler_type)

In [4]:
# Train model
# -----------
x_train = data_train[featuresNA].values
y_train = data_train[phenoNA].values

x_test = data_test[featuresNA].values
y_test = data_test[phenoNA].values

print('\n* TRAINING MODEL:')
model = train_model(model_name, x_train, y_train, x_test, y_test)


* TRAINING MODEL:
Model accuracy: Train: 0.69, Test: 0.72
Model balanced accuracy: Train: 0.68, Test: 0.69



## Tensorize

In [5]:
# Tensorize model
# ---------------
xt_train = torch.from_numpy(x_train).float()
yt_train = torch.from_numpy(y_train)

xt_test = torch.from_numpy(x_test).float()
yt_test = torch.from_numpy(y_test)

# sketch_size could be chosen smaller, e.g., sketch_size = 50, and the
# tensorization will be much faster returning a tn_model with the same accuracy,
# but sometimes this could lead to having negative values for inputs with low
# probability. Choosing a bigger sketch_size usually avoids these errors.
sketch_size    = 200
phys_dim       = 2
domain         = torch.linspace(0, 1, phys_dim)
bond_dim       = 2
cum_percentage = 1 - 1e-5
batch_size     = 500
device         = torch.device('cpu')
dtype          = torch.float
verbose        = False

print('* TENSORIZING MODEL:')
tn_model = tensorize(model=model,
                     x_train=xt_train,
                     y_train=yt_train,
                     x_test=xt_test,
                     y_test=yt_test,
                     sketch_size=sketch_size,
                     phys_dim=phys_dim,
                     domain=domain,
                     bond_dim=bond_dim,
                     cum_percentage=cum_percentage,
                     batch_size=batch_size,
                     device=device,
                     dtype=dtype,
                     verbose=verbose)

* TENSORIZING MODEL:
Info: {'total_time': 0.9849131107330322, 'val_eps': tensor(0.0407)}
MSE: Train: 0.00085, Test: 9.81e-04
tensor([[0.6774, 0.3227],
        [0.4294, 0.5875],
        [0.6202, 0.3891],
        [0.4023, 0.6144],
        [0.8708, 0.1735],
        [0.6338, 0.3380],
        [0.6911, 0.3142],
        [0.3829, 0.6272],
        [0.5891, 0.4170],
        [0.7322, 0.2631]])
tensor([[0.6911, 0.3089],
        [0.4131, 0.5869],
        [0.6299, 0.3701],
        [0.3812, 0.6188],
        [0.8390, 0.1610],
        [0.6398, 0.3602],
        [0.7020, 0.2980],
        [0.3694, 0.6306],
        [0.5971, 0.4029],
        [0.7406, 0.2594]])
Model accuracy: Train: 0.70, Test: 0.72
Model balanced accuracy: Train: 0.68, Test: 0.69



In [6]:
# Renormalize model
# -----------------

# We renormalize the tensor network so that it represents a normalized
# distribution of all the features, instead of only giving normalized
# conditional distributions for the output feature ("Response")

# Contiuous variables have to be integrated. "discr_steps" indicates the number
# of dicretization steps used to discretize the continuous variable and perform
# numerical integration
discr_steps = int(1e5)

print('* RENORMALIZING TN MODEL:')
tn_model = renormalize(mps=tn_model,
                       phys_dim=phys_dim,
                       discr_steps=discr_steps,
                       n_classes=n_classes,
                       num_features=numeric_featuresNA,
                       x_train=xt_train,
                       x_test=xt_test,
                       y_test=yt_test)

* RENORMALIZING TN MODEL:
Model accuracy: Test: 0.72
Model balanced accuracy: Test: 0.69



In [7]:
def fn_model(data):
    result = torch.from_numpy(model.predict_proba(data)).float()
    return result

def embedding(data):
    return tk.embeddings.poly(data, degree=phys_dim - 1).float()

In [8]:
fn_model(xt_test)

tensor([[0.2541, 0.7459],
        [0.4482, 0.5518],
        [0.5097, 0.4903],
        ...,
        [0.5680, 0.4320],
        [0.7511, 0.2489],
        [0.5113, 0.4887]])

In [9]:
tn_model(embedding(xt_test))

tensor([[2.0991e-06, 5.3931e-06],
        [3.5108e-06, 4.1144e-06],
        [3.9200e-06, 3.7436e-06],
        ...,
        [4.3015e-06, 3.3516e-06],
        [5.6476e-06, 1.8608e-06],
        [3.9968e-06, 3.6940e-06]], grad_fn=<ViewBackward0>)

In [10]:
tn_model(embedding(xt_test)) / tn_model(embedding(xt_test)).sum(dim=1, keepdim=True)

tensor([[0.2802, 0.7198],
        [0.4604, 0.5396],
        [0.5115, 0.4885],
        ...,
        [0.5621, 0.4379],
        [0.7522, 0.2478],
        [0.5197, 0.4803]], grad_fn=<DivBackward0>)

## Study distributions

### Examples

In [11]:
# Get conditional/marginal distribution
# -------------------------------------

# We need to specify the condition and marginal variables. The rest will be
# marginalized out. The "Response" feature can be included as condition or
# marginal

# Select features on which we condition
cond_features = ['TMB', 'Albumin']

# Create tensor with values (already scaled to [0, 1])
# for conditioned features
cond_data = [0.2, 0.9]  # Just an example

# Select features which we marginalize
marg_features = ['Systemic_therapy_history', 'Response']

# In this example, we will get the distribution 
# P('Systemic_therapy_history', 'Response' | 'TMB', 'Albumin'),
# marginalizing out the rest of the features

# The distribution will be given as a tensor with shape:
# (dim_marg_1, ..., dim_marg_n), with "dim_marg_{i}" being the dimensions of
# each marginal feature. These marginal features could be returned in a
# different order as the one specified in "marg_features". The order will
# be returned in the list "marg_feat_order"

# For continuous features, the dimension of the marginal features will be the
# specified "discr_steps"

discr_steps = int(1e5)

distr, marg_feat_order = get_distribution(
    mps=tn_model,
    cond_features=cond_features,
    cond_data=cond_data,
    marg_features=marg_features,
    in_features=featuresNA,
    out_feature=phenoNA,
    num_features=numeric_featuresNA,
    n_classes=n_classes,
    phys_dim=phys_dim,
    x_train=xt_train,
    discr_steps=discr_steps
)

print(marg_feat_order, distr.shape)
print(distr)
print(distr.sum())
print(marg_feat_order[1], distr.sum(dim=0))
print(marg_feat_order[0], distr.sum(dim=1))
print()

['Systemic_therapy_history', 'Response'] torch.Size([2, 2])
tensor([[0.2649, 0.2357],
        [0.3095, 0.1899]])
tensor(1.)
Response tensor([0.5744, 0.4256])
Systemic_therapy_history tensor([0.5007, 0.4993])



In [12]:
# Example 2
cond_features = ['Response']
cond_data = [1.]

marg_features = ['TMB']

discr_steps = int(1e1)  # int(1e5)

distr, marg_feat_order = get_distribution(
    mps=tn_model,
    cond_features=cond_features,
    cond_data=cond_data,
    marg_features=marg_features,
    in_features=featuresNA,
    out_feature=phenoNA,
    num_features=numeric_featuresNA,
    n_classes=n_classes,
    phys_dim=phys_dim,
    x_train=xt_train,
    discr_steps=discr_steps
)

print(marg_feat_order, distr.shape)
print(distr)
print(distr.sum())
print()

['TMB'] torch.Size([10])
tensor([0.0462, 0.0582, 0.0701, 0.0821, 0.0940, 0.1060, 0.1179, 0.1299, 0.1418,
        0.1538])
tensor(1.)



In [13]:
# Example 3
cond_features = ['Systemic_therapy_history']
cond_data = [0.]

marg_features = ['Response']

discr_steps = int(1e5)

distr, marg_feat_order = get_distribution(
    mps=tn_model,
    cond_features=cond_features,
    cond_data=cond_data,
    marg_features=marg_features,
    in_features=featuresNA,
    out_feature=phenoNA,
    num_features=numeric_featuresNA,
    n_classes=n_classes,
    phys_dim=phys_dim,
    x_train=xt_train,
    discr_steps=discr_steps
)

print(marg_feat_order, distr.shape)
print(distr)
print(distr.sum())
print()

['Response'] torch.Size([2])
tensor([0.5385, 0.4615])
tensor(1.0000)



In [14]:
# Example 4
cond_features = []
cond_data = []

marg_features = ['Response', 'Albumin']

discr_steps = int(1e1)  # int(1e5)

distr, marg_feat_order = get_distribution(
    mps=tn_model,
    cond_features=cond_features,
    cond_data=cond_data,
    marg_features=marg_features,
    in_features=featuresNA,
    out_feature=phenoNA,
    num_features=numeric_featuresNA,
    n_classes=n_classes,
    phys_dim=phys_dim,
    x_train=xt_train,
    discr_steps=discr_steps
)

print(marg_feat_order, distr.shape)
print(distr / distr.sum(dim=1, keepdim=True))
print()

['Albumin', 'Response'] torch.Size([10, 2])
tensor([[0.7730, 0.2270],
        [0.7311, 0.2689],
        [0.6890, 0.3110],
        [0.6469, 0.3531],
        [0.6048, 0.3952],
        [0.5625, 0.4375],
        [0.5203, 0.4797],
        [0.4779, 0.5221],
        [0.4355, 0.5645],
        [0.3931, 0.6069]])



### Feature Importance

In [ ]:
# TMB: We know TMB above 27 indicates better response

# NOTE: I need to pass all arguments to marginal_prediciton.
# Needed? Or just use get_distribution?

low = marginal_prediction(5, ['TMB'], scalers_train, xt_train)
mid = marginal_prediction(27, ['TMB'], scalers_train, xt_train)
high = marginal_prediction(50, ['TMB'], scalers_train, xt_train)

print('TMB Example:')
print(f'TT Score: {low, mid, high}')
print()

NameError: name 'tn_model' is not defined